# Dirección Directiva

> **Contexto de dominio:** [`docs/fuentes/direccion_directiva.md`](../../docs/fuentes/direccion_directiva.md)  
> **Bloque:** A | **Línea:** `direccion_directiva`  
> **Variable principal:** `indicador` ()  
> **Modelos sugeridos:** RANDOM_FOREST, XGBOOST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/direccion_directiva.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "direccion_directiva"
VARIABLE = "indicador"
UNIDAD = ""

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `direccion_directiva` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "direccion_directiva.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/direccion_directiva.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/direccion_directiva.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "indicador": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

<!-- ENRICHMENT: direccion_directiva -->

## 1b. Marco DPSIR y dashboard ICAU — Analitica institucional

El marco **DPSIR** (Fuerzas Motrices - Presion - Estado - Impacto - Respuesta), promovido por la OCDE y adoptado por el IDEAM, estructura la causalidad ambiental:

| Componente | Ejemplo ambiental | Indicador |
|---|---|---|
| **D** Fuerzas motrices | Crecimiento urbano, industria | Densidad poblacional, PIB |
| **P** Presion | Vertimientos, emisiones, deforestacion | IACAL, RETC, TCCN |
| **S** Estado | Calidad aire/agua, cobertura bosque | ICA, IVH, IRH |
| **I** Impacto | Salud publica, perdida biodiversidad | Morbilidad, BMWP-Col |
| **R** Respuesta | Normas, inversiones, PUEAA | % POT actualizados, IEDI |

**ICAU (Indice de Calidad Ambiental Urbana):** agrega 6 componentes con pesos iguales:
aire (PM2.5/PM10), agua (ICA), espacio publico, residuos, arbolado y ruido.

**IEDI (Indice de Evaluacion del Desempeño Institucional):** mide la gestion de las CAR en 3 ejes: misional (eficacia/eficiencia), financiero y administrativo. Res. 667/2016.

In [ ]:
# Marco DPSIR sintetico + ICAU por ciudad + IEDI por CAR
np.random.seed(88)
n = len(df)

# -- DPSIR: variables por componente (indices normalizados 0-1) -----------
dpsir_data = {
    'Fuerzas_Motrices': 0.72,  # densidad pob + PIB industrial
    'Presion':          0.65,  # IACAL + emisiones + deforest
    'Estado':           0.41,  # ICA + IRH + cobertura bosque
    'Impacto':          0.55,  # morbilidad + biodiversidad
    'Respuesta':        0.38,  # % POT actualizados + IEDI promedio
}

# -- ICAU por ciudad (6 componentes) ------------------------------------
ciudades = ['Bogota','Medellin','Cali','Barranquilla','Bucaramanga','Manizales']
componentes_icau = ['Aire','Agua','Espacio_publico','Residuos','Arbolado','Ruido']
icau_matrix = np.random.uniform(0.3, 0.9, (len(ciudades), len(componentes_icau)))
icau_total = icau_matrix.mean(axis=1)  # promedio simple
df_icau = pd.DataFrame(icau_matrix, columns=componentes_icau, index=ciudades)
df_icau['ICAU_total'] = icau_total.round(3)

# -- IEDI por CAR (escala 0-100) ----------------------------------------
cars = ['CAR','CORPOAMAZONIA','CORNARE','CVC','CORANTIOQUIA','CORPOCESAR']
iedi_misional  = np.random.uniform(50, 90, len(cars))
iedi_financiero= np.random.uniform(40, 85, len(cars))
iedi_admin     = np.random.uniform(45, 88, len(cars))
iedi_total     = (iedi_misional * 0.5 + iedi_financiero * 0.3 + iedi_admin * 0.2)
df_iedi = pd.DataFrame({'CAR': cars, 'Misional': iedi_misional.round(1),
                         'Financiero': iedi_financiero.round(1),
                         'Administrativo': iedi_admin.round(1),
                         'IEDI_total': iedi_total.round(1)})

print('Marco DPSIR (escala 0-1, mayor = mayor presion/impacto):')
for k, v in dpsir_data.items():
    bar = '|' * int(v * 20)
    print(f'  {k:20s}: {bar} {v:.2f}')
print(f'\nICAU promedio ciudades: {icau_total.mean():.3f}')
print(f'Ciudad mejor ICAU: {ciudades[icau_total.argmax()]} ({icau_total.max():.3f})')
print(f'IEDI promedio CARs: {iedi_total.mean():.1f}/100')
df_iedi

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `direccion_directiva` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_direccion_directiva.html",
       title="EDA — Dirección Directiva", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "indicador", title="Dirección Directiva — indicador ()")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "indicador", period="month")
plt.show()

## 3c. Dashboard ICAU — Calidad Ambiental Urbana por ciudad y componente

El ICAU permite a los directivos comparar el desempeño ambiental de ciudades y priorizar inversiones en el componente mas deficitario. La Resolucion 667/2016 (MADS) establece los **Indicadores Minimos de Gestion (IMG)** que las CAR deben reportar anualmente.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: ICAU por componente y ciudad (heatmap)
ax = axes[0]
im = ax.imshow(df_icau[componentes_icau].values, cmap='RdYlGn',
               aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(componentes_icau)))
ax.set_xticklabels(componentes_icau, rotation=30, ha='right', fontsize=8)
ax.set_yticks(range(len(ciudades)))
ax.set_yticklabels(ciudades, fontsize=9)
for i in range(len(ciudades)):
    for j in range(len(componentes_icau)):
        ax.text(j, i, f'{df_icau[componentes_icau].values[i,j]:.2f}',
                ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax, label='ICAU componente')
ax.set_title('ICAU por Componente y Ciudad\n(Res. 667/2016 — IMG MADS)', fontweight='bold')

# Panel B: IEDI por CAR
ax = axes[1]
x = range(len(cars))
width = 0.25
ax.bar([i - width for i in x], df_iedi['Misional'], width, label='Misional (50%)', color='#27ae60', alpha=0.85)
ax.bar(x, df_iedi['Financiero'], width, label='Financiero (30%)', color='#3498db', alpha=0.85)
ax.bar([i + width for i in x], df_iedi['Administrativo'], width, label='Administrativo (20%)', color='#e67e22', alpha=0.85)
ax.plot(x, df_iedi['IEDI_total'], 'ko-', ms=6, lw=2, label='IEDI total')
ax.set_xticks(x); ax.set_xticklabels(cars, rotation=20, ha='right', fontsize=8)
ax.set_title('IEDI por CAR — Evaluacion Desempeño Institucional', fontweight='bold')
ax.set_ylabel('Puntaje (0-100)'); ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()
print(f'CAR mejor IEDI: {df_iedi.loc[df_iedi["IEDI_total"].idxmax(), "CAR"]} '
      f'({df_iedi["IEDI_total"].max():.1f}/100)')

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["indicador"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas
> Compara `indicador` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["indicador"], variable="indicador")
if rep.empty:
    print("Sin normas colombianas registradas para 'indicador'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["indicador"], method="linear")
print(f"Faltantes antes: {df["indicador"].isna().sum()} | "
      f"después: {df_clean["indicador"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["indicador"]

models = {
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Isolation Forest — Deteccion de anomalias en ejecucion presupuestal

El **Isolation Forest** es un algoritmo no supervisado que detecta outliers aislando puntos atipicos en pocas particiones del arbol. En gestion publica ambiental se usa para:

- Detectar contratos con valores atipicos (posibles irregularidades en contratacion)
- Identificar CARs con ejecucion presupuestal anormalmente baja o alta
- Anomalias en reportes de indicadores (posible subregistro o error de digitacion)

```python
from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.1, random_state=42)
anomalias = iso.fit_predict(X)   # -1 = anomalia, +1 = normal
scores = iso.decision_function(X)  # scores mas negativos = mas anomalo
```

In [ ]:
from sklearn.ensemble import IsolationForest

np.random.seed(42)
N = 120  # contratos/proyectos simulados

# Features: valor contrato (M$), duracion (meses), ejecucion (%)
valor = np.random.lognormal(mean=3.5, sigma=0.8, size=N)     # M pesos
duracion = np.random.randint(3, 36, N)                         # meses
ejecucion = np.clip(np.random.normal(75, 15, N), 0, 100)      # %

# Inyectar anomalias (5 contratos irregulares)
idx_anoms = np.random.choice(N, 5, replace=False)
valor[idx_anoms] = np.random.uniform(500, 2000, 5)  # valores extremos
ejecucion[idx_anoms] = np.random.uniform(0, 10, 5)  # ejecucion casi cero

X_if = np.column_stack([valor, duracion, ejecucion])

iso = IsolationForest(contamination=0.08, random_state=42, n_estimators=100)
pred_if = iso.fit_predict(X_if)   # -1=anomalia
scores_if = iso.decision_function(X_if)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Panel A: Valor vs Ejecucion coloreado por anomalia
colors_if = ['#e74c3c' if p == -1 else '#2980b9' for p in pred_if]
ax1.scatter(valor, ejecucion, c=colors_if, alpha=0.7, s=30)
ax1.set_xlabel('Valor contrato (M pesos)')
ax1.set_ylabel('Ejecucion presupuestal (%)')
ax1.set_title('Isolation Forest — Anomalias presupuestales CAR', fontweight='bold')
from matplotlib.patches import Patch
ax1.legend(handles=[Patch(color='#e74c3c',label='Anomalia'),
                     Patch(color='#2980b9',label='Normal')], fontsize=8)
ax1.grid(alpha=0.3)

# Panel B: Scores de anomalia
sorted_idx = np.argsort(scores_if)
ax2.barh(range(15), scores_if[sorted_idx[:15]], color=[
    '#e74c3c' if pred_if[i]==-1 else '#2980b9' for i in sorted_idx[:15]])
ax2.axvline(0, color='black', lw=1, ls='--')
ax2.set_title('Top 15 scores Isolation Forest\n(negativo = mas anomalo)', fontweight='bold')
ax2.set_xlabel('Anomaly score'); ax2.grid(axis='x', alpha=0.3)

plt.tight_layout(); plt.show()
n_anom = (pred_if == -1).sum()
print(f'Contratos anomalos detectados: {n_anom}/{N} ({n_anom/N*100:.1f}%)')
print(f'Contratos inyectados como irregulares: {len(idx_anoms)}')
print('Usar en produccion con datos SECOP II o SUI para auditoria presupuestal')

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: direccion_directiva -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Dirección directiva / KPIs

- **KPIs ejecutivos:** indicadores agregados — series mensuales/trimestrales, no diarias/horarias.
- **Comparabilidad inter-período:** usar misma metodología en todas las mediciones; documentar cambios.
- **Reporting trimestral / anual** — evitar conclusiones operativas sobre datos de gestión.

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** Dirección Directiva (`direccion_directiva`)
- **Variable analizada:** `indicador` ()
- **Modelos ejecutados:** RANDOM_FOREST, XGBOOST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/direccion_directiva.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.